# Data Access

In [0]:
storage_account = "nastorage003"

adls_oauth_configs = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": "<YOUR_AZURE_CLIENT_ID>",
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": "<YOUR_AZURE_CLIENT_SECRET>",
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": "https://login.microsoftonline.com/<YOUR_AZURE_TENANT_ID>/oauth2/token",
}

In [0]:
for key, value in adls_oauth_configs.items():
    spark.conf.set(key, value)

dbutils.fs.ls("abfss://churnml@nastorage003.dfs.core.windows.net/")

[FileInfo(path='abfss://churnml@nastorage003.dfs.core.windows.net/churn_feature_table/', name='churn_feature_table/', size=0, modificationTime=1780079519000)]

# Read Table

In [0]:
from pyspark.sql import functions as F

ML_BASE_PATH = "abfss://churnml@nastorage003.dfs.core.windows.net"
CHURN_FEATURE_PATH = f"{ML_BASE_PATH}/churn_feature_table"

churn_df = (
    spark.read
    .format("delta")
    .load(CHURN_FEATURE_PATH)
)

display(churn_df.limit(5))

customer_id,region,customer_type,plan_type,contract_type,monthly_charge_aud,tenure_months,has_cybersecurity_addon,has_voip_addon,avg_daily_bandwidth_gb,avg_peak_latency_ms,avg_offpeak_latency_ms,avg_packet_loss_pct,avg_connection_drops,outage_days,sla_breach_days,ticket_count,ticket_resolved_rate,avg_ticket_satisfaction,avg_resolution_days,billing_cycles,avg_amount_charged,late_rate,failed_rate,churn_flag,churn_date
CUST-00833,Sydney,residential,NBN50,month-to-month,57.63,24,false,false,7.990666666666665,13.595555555555555,7.522222222222222,0.37544444444444436,0.32222222222222224,3,2,1,0.0,null,null,12,58.02333333333333,0.0,0.0,false,null
CUST-01238,Brisbane,residential,NBN100,month-to-month,77.68,1,false,false,17.534777777777773,15.468888888888891,8.778888888888888,0.3303333333333333,0.35555555555555557,2,1,1,1.0,1.0,12.0,12,77.77083333333333,0.08333333333333333,0.08333333333333333,false,null
CUST-01084,Melbourne,residential,NBN250,2-year,100.14,11,false,true,37.89533333333333,11.911111111111111,6.736666666666666,0.40777777777777774,0.35555555555555557,3,2,3,0.6666666666666666,4.5,5.5,12,99.62333333333333,0.08333333333333333,0.0,false,null
CUST-01395,Brisbane,wholesale,Fibre,2-year,297.53,13,false,false,152.76900000000003,14.937777777777773,8.597777777777777,0.39566666666666667,0.37777777777777777,2,2,2,1.0,5.0,5.0,12,296.9883333333334,0.0,0.0,false,null
CUST-01990,Brisbane,residential,NBN50,2-year,57.4,30,false,false,8.469777777777777,14.80111111111111,8.016666666666667,0.4485555555555556,0.4111111111111111,2,1,3,1.0,3.3333333333333335,9.0,12,57.845,0.0,0.0,false,null


# Preprocessing

In [0]:
# Drop rows where label is null
churn_df_clean = churn_df.filter(churn_df.churn_flag.isNotNull())


# Cast label to integer (0/1)
churn_df_clean = churn_df_clean.withColumn(
    "label",
    F.col("churn_flag").cast("int")
)

display(churn_df_clean.limit(5))

customer_id,region,customer_type,plan_type,contract_type,monthly_charge_aud,tenure_months,has_cybersecurity_addon,has_voip_addon,avg_daily_bandwidth_gb,avg_peak_latency_ms,avg_offpeak_latency_ms,avg_packet_loss_pct,avg_connection_drops,outage_days,sla_breach_days,ticket_count,ticket_resolved_rate,avg_ticket_satisfaction,avg_resolution_days,billing_cycles,avg_amount_charged,late_rate,failed_rate,churn_flag,churn_date,label
CUST-00833,Sydney,residential,NBN50,month-to-month,57.63,24,false,false,7.990666666666665,13.595555555555555,7.522222222222222,0.37544444444444436,0.32222222222222224,3,2,1,0.0,null,null,12,58.02333333333333,0.0,0.0,false,null,0
CUST-01238,Brisbane,residential,NBN100,month-to-month,77.68,1,false,false,17.534777777777773,15.468888888888891,8.778888888888888,0.3303333333333333,0.35555555555555557,2,1,1,1.0,1.0,12.0,12,77.77083333333333,0.08333333333333333,0.08333333333333333,false,null,0
CUST-01084,Melbourne,residential,NBN250,2-year,100.14,11,false,true,37.89533333333333,11.911111111111111,6.736666666666666,0.40777777777777774,0.35555555555555557,3,2,3,0.6666666666666666,4.5,5.5,12,99.62333333333333,0.08333333333333333,0.0,false,null,0
CUST-01395,Brisbane,wholesale,Fibre,2-year,297.53,13,false,false,152.76900000000003,14.937777777777773,8.597777777777777,0.39566666666666667,0.37777777777777777,2,2,2,1.0,5.0,5.0,12,296.9883333333334,0.0,0.0,false,null,0
CUST-01990,Brisbane,residential,NBN50,2-year,57.4,30,false,false,8.469777777777777,14.80111111111111,8.016666666666667,0.4485555555555556,0.4111111111111111,2,1,3,1.0,3.3333333333333335,9.0,12,57.845,0.0,0.0,false,null,0


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# Cast booleans to numeric 0/1 for ML
churn_ml_df = (
    churn_df_clean
    .withColumn("has_cybersecurity_addon_num", F.col("has_cybersecurity_addon").cast("int"))
    .withColumn("has_voip_addon_num",          F.col("has_voip_addon").cast("int"))
)


In [0]:
from pyspark.sql import functions as F

# 1) Define which columns to use
categorical_cols = ["customer_type", "plan_type", "contract_type", "region"]

numeric_cols = [
    "monthly_charge_aud",
    "tenure_months",
    "has_cybersecurity_addon_num",
    "has_voip_addon_num",
    "avg_daily_bandwidth_gb",
    "avg_peak_latency_ms",
    "avg_offpeak_latency_ms",
    "avg_packet_loss_pct",
    "avg_connection_drops",
    "outage_days",
    "sla_breach_days",
    "ticket_count",
    "ticket_resolved_rate",
    "avg_ticket_satisfaction",
    "avg_resolution_days",
    "billing_cycles",
    "avg_amount_charged",
    "late_rate",
    "failed_rate"
]

feature_cols = categorical_cols + numeric_cols
all_cols = feature_cols + ["label", "customer_id", "region"]

# 2) Convert to pandas
pandas_df = churn_ml_df.select(*all_cols).toPandas()

# 3) One-hot encode categoricals in pandas
import pandas as pd

X = pd.get_dummies(
    pandas_df[feature_cols],
    columns=categorical_cols,
    drop_first=True  # avoid redundant dummy columns
)

y = pandas_df["label"]

In [0]:
X.head()

,monthly_charge_aud,tenure_months,has_cybersecurity_addon_num,has_voip_addon_num,avg_daily_bandwidth_gb,avg_peak_latency_ms,avg_offpeak_latency_ms,avg_packet_loss_pct,avg_connection_drops,outage_days,sla_breach_days,ticket_count,ticket_resolved_rate,avg_ticket_satisfaction,avg_resolution_days,billing_cycles,avg_amount_charged,late_rate,failed_rate,customer_type_residential,customer_type_wholesale,plan_type_Fibre,plan_type_NBN100,plan_type_NBN1000,plan_type_NBN250,plan_type_NBN50,plan_type_Wholesale_Dark_Fibre,contract_type_2-year,contract_type_month-to-month,region_Brisbane,region_Canberra,region_Melbourne,region_Perth,region_Sydney,region_Brisbane,region_Canberra,region_Melbourne,region_Perth,region_Sydney
0,57.63,24,0,0,7.990667,13.595556,7.522222,0.375444,0.322222,3.0,2.0,1.0,0.000000,NaN,NaN,12.0,58.023333,0.000000,0.000000,1,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,1
1,77.68,1,0,0,17.534778,15.468889,8.778889,0.330333,0.355556,2.0,1.0,1.0,1.000000,1.000000,12.0,12.0,77.770833,0.083333,0.083333,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0
2,100.14,11,0,1,37.895333,11.911111,6.736667,0.407778,0.355556,3.0,2.0,3.0,0.666667,4.500000,5.5,12.0,99.623333,0.083333,0.000000,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0
3,297.53,13,0,0,152.769000,14.937778,8.597778,0.395667,0.377778,2.0,2.0,2.0,1.000000,5.000000,5.0,12.0,296.988333,0.000000,0.000000,0,1,1,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0
4,57.40,30,0,0,8.469778,14.801111,8.016667,0.448556,0.411111,2.0,1.0,3.0,1.000000,3.333333,9.0,12.0,57.845000,0.000000,0.000000,1,0,0,0,0,0,1,0,1,0,1,0,0,0,0,1,0,0,0,0


In [0]:
from sklearn.impute import SimpleImputer
import numpy as np

# 1) Impute missing numeric values with median
imputer = SimpleImputer(strategy="median")

X = imputer.fit_transform(X)

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1) Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y   # keep churn ratio similar in train and test
)

# 2) Define and train logistic regression model
lr = LogisticRegression(
    max_iter=2000,    # allow more iterations
    solver="lbfgs",
    C=1.0             # standard L2 regularisation strength
)

lr.fit(X_train, y_train)

y_pred_proba = lr.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, y_pred_proba)
print(f"Logistic Regression AUC: {lr_auc:.3f}")

Logistic Regression AUC: 0.869


In [0]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score


# 2) Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb.fit(X_train, y_train)
gb_proba = gb.predict_proba(X_test)[:, 1]
gb_auc = roc_auc_score(y_test, gb_proba)
print(f"Gradient Boosting AUC: {gb_auc:.3f}")

# 3) Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_proba)
print(f"Random Forest AUC: {rf_auc:.3f}")


Gradient Boosting AUC: 1.000
Random Forest AUC: 1.000


In [0]:
import pandas as pd

# Create a small DataFrame with label and a few strongest-looking features
leak_check = pandas_df[[
    "label",
    "tenure_months",
    "ticket_count",
    "ticket_resolved_rate",
    "avg_ticket_satisfaction",
    "late_rate",
    "failed_rate",
    "sla_breach_days"
]]

leak_check.groupby("label").mean()

,tenure_months,ticket_count,ticket_resolved_rate,avg_ticket_satisfaction,late_rate,failed_rate,sla_breach_days
label,,,,,,,
0,21.684118,1.925758,0.793914,3.249704,0.100343,0.049608,2.114458
1,19.050000,1.948936,0.786312,3.240421,0.080037,0.039474,NaN


In [0]:
import pandas as pd

auc_table = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "Gradient Boosting",
        "Random Forest"
    ],
    "auc": [
        lr_auc,
        gb_auc,
        rf_auc
    ]
})

auc_table

,model,auc
0,Logistic Regression,0.868627
1,Gradient Boosting,1.000000
2,Random Forest,0.999902


In [0]:
# Remove duplicate columns, keeping the first occurrence
pandas_df = pandas_df.loc[:, ~pandas_df.columns.duplicated()]

print(pandas_df.columns)

Index(['customer_type', 'plan_type', 'contract_type', 'region',
       'monthly_charge_aud', 'tenure_months', 'has_cybersecurity_addon_num',
       'has_voip_addon_num', 'avg_daily_bandwidth_gb', 'avg_peak_latency_ms',
       'avg_offpeak_latency_ms', 'avg_packet_loss_pct', 'avg_connection_drops',
       'outage_days', 'sla_breach_days', 'ticket_count',
       'ticket_resolved_rate', 'avg_ticket_satisfaction',
       'avg_resolution_days', 'billing_cycles', 'avg_amount_charged',
       'late_rate', 'failed_rate', 'label', 'customer_id',
       'churn_probability'],
      dtype='object')


In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 0) (Safety) remove any duplicate-named columns from pandas_df
pandas_df = pandas_df.loc[:, ~pandas_df.columns.duplicated()]

# 1) Refit LR on full dataset for final model
lr_final = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    C=1.0
)

lr_final.fit(X, y)

# 2) Training AUC on all data
y_all_proba = lr_final.predict_proba(X)[:, 1]
print("Training AUC (all data):", roc_auc_score(y, y_all_proba))

# 3) Attach probabilities back to pandas_df
pandas_df["churn_probability"] = y_all_proba

# 4) Build churn scores table with rich filters
scores_pdf = pandas_df[[
    "customer_id",
    "customer_type",
    "plan_type",
    "contract_type",
    "region",
    "tenure_months",
    "monthly_charge_aud",
    "avg_daily_bandwidth_gb",
    "avg_peak_latency_ms",
    "avg_offpeak_latency_ms",
    "avg_packet_loss_pct",
    "avg_connection_drops",
    "outage_days",
    "sla_breach_days",
    "ticket_count",
    "ticket_resolved_rate",
    "avg_ticket_satisfaction",
    "billing_cycles",
    "avg_amount_charged",
    "late_rate",
    "failed_rate",
    "label",
    "churn_probability"
]].copy()

# 5) Rename label for clarity
scores_pdf = scores_pdf.rename(columns={"label": "churn_flag"})

scores_pdf.head()

Training AUC (all data): 0.8738254901960785


,customer_id,customer_type,plan_type,contract_type,region,tenure_months,monthly_charge_aud,avg_daily_bandwidth_gb,avg_peak_latency_ms,avg_offpeak_latency_ms,avg_packet_loss_pct,avg_connection_drops,outage_days,sla_breach_days,ticket_count,ticket_resolved_rate,avg_ticket_satisfaction,billing_cycles,avg_amount_charged,late_rate,failed_rate,churn_flag,churn_probability
0,CUST-00833,residential,NBN50,month-to-month,Sydney,24,57.63,7.990667,13.595556,7.522222,0.375444,0.322222,3.0,2.0,1.0,0.000000,NaN,12.0,58.023333,0.000000,0.000000,0,0.392714
1,CUST-01238,residential,NBN100,month-to-month,Brisbane,1,77.68,17.534778,15.468889,8.778889,0.330333,0.355556,2.0,1.0,1.0,1.000000,1.000000,12.0,77.770833,0.083333,0.083333,0,0.042026
2,CUST-01084,residential,NBN250,2-year,Melbourne,11,100.14,37.895333,11.911111,6.736667,0.407778,0.355556,3.0,2.0,3.0,0.666667,4.500000,12.0,99.623333,0.083333,0.000000,0,0.043454
3,CUST-01395,wholesale,Fibre,2-year,Brisbane,13,297.53,152.769000,14.937778,8.597778,0.395667,0.377778,2.0,2.0,2.0,1.000000,5.000000,12.0,296.988333,0.000000,0.000000,0,0.004180
4,CUST-01990,residential,NBN50,2-year,Brisbane,30,57.40,8.469778,14.801111,8.016667,0.448556,0.411111,2.0,1.0,3.0,1.000000,3.333333,12.0,57.845000,0.000000,0.000000,0,0.009101


In [0]:
# Convert to Spark
scores_sdf = spark.createDataFrame(scores_pdf)

print(scores_sdf.columns)   # sanity check: all unique names

ML_BASE_PATH = "abfss://churnml@nastorage003.dfs.core.windows.net/ml"
CHURN_SCORES_PATH = f"{ML_BASE_PATH}/churn_scores"

(
    scores_sdf
    .write
    .mode("overwrite")
    .format("delta")
    .save(CHURN_SCORES_PATH)
)

display(scores_sdf.limit(5))

['customer_id', 'customer_type', 'plan_type', 'contract_type', 'region', 'tenure_months', 'monthly_charge_aud', 'avg_daily_bandwidth_gb', 'avg_peak_latency_ms', 'avg_offpeak_latency_ms', 'avg_packet_loss_pct', 'avg_connection_drops', 'outage_days', 'sla_breach_days', 'ticket_count', 'ticket_resolved_rate', 'avg_ticket_satisfaction', 'billing_cycles', 'avg_amount_charged', 'late_rate', 'failed_rate', 'churn_flag', 'churn_probability']


customer_id,customer_type,plan_type,contract_type,region,tenure_months,monthly_charge_aud,avg_daily_bandwidth_gb,avg_peak_latency_ms,avg_offpeak_latency_ms,avg_packet_loss_pct,avg_connection_drops,outage_days,sla_breach_days,ticket_count,ticket_resolved_rate,avg_ticket_satisfaction,billing_cycles,avg_amount_charged,late_rate,failed_rate,churn_flag,churn_probability
CUST-00833,residential,NBN50,month-to-month,Sydney,24,57.63,7.990666666666665,13.595555555555555,7.522222222222222,0.37544444444444436,0.32222222222222224,3.0,2.0,1.0,0.0,null,12.0,58.02333333333333,0.0,0.0,0,0.3927144663026507
CUST-01238,residential,NBN100,month-to-month,Brisbane,1,77.68,17.534777777777773,15.468888888888891,8.778888888888888,0.3303333333333333,0.35555555555555557,2.0,1.0,1.0,1.0,1.0,12.0,77.77083333333333,0.08333333333333333,0.08333333333333333,0,0.04202612159896645
CUST-01084,residential,NBN250,2-year,Melbourne,11,100.14,37.89533333333333,11.911111111111111,6.736666666666666,0.40777777777777774,0.35555555555555557,3.0,2.0,3.0,0.6666666666666666,4.5,12.0,99.62333333333333,0.08333333333333333,0.0,0,0.04345363469363999
CUST-01395,wholesale,Fibre,2-year,Brisbane,13,297.53,152.76900000000003,14.937777777777773,8.597777777777777,0.39566666666666667,0.37777777777777777,2.0,2.0,2.0,1.0,5.0,12.0,296.9883333333334,0.0,0.0,0,0.004180083354007641
CUST-01990,residential,NBN50,2-year,Brisbane,30,57.4,8.469777777777777,14.80111111111111,8.016666666666667,0.4485555555555556,0.4111111111111111,2.0,1.0,3.0,1.0,3.3333333333333335,12.0,57.845,0.0,0.0,0,0.009101191033806197


In [0]:
# 2) Also register as a managed table in the warehouse
#    (adjust database/schema name if needed)
# Register table in the isp.ml schema

spark.sql("""
CREATE SCHEMA IF NOT EXISTS isp_ml
""")

scores_sdf.write.mode("overwrite").saveAsTable("isp_ml.churn_scores")

# Quick check from Spark SQL
display(spark.sql("SELECT * FROM isp_ml.churn_scores LIMIT 5"))

customer_id,customer_type,plan_type,contract_type,region,tenure_months,monthly_charge_aud,avg_daily_bandwidth_gb,avg_peak_latency_ms,avg_offpeak_latency_ms,avg_packet_loss_pct,avg_connection_drops,outage_days,sla_breach_days,ticket_count,ticket_resolved_rate,avg_ticket_satisfaction,billing_cycles,avg_amount_charged,late_rate,failed_rate,churn_flag,churn_probability
CUST-00833,residential,NBN50,month-to-month,Sydney,24,57.63,7.990666666666665,13.595555555555555,7.522222222222222,0.37544444444444436,0.32222222222222224,3.0,2.0,1.0,0.0,null,12.0,58.02333333333333,0.0,0.0,0,0.3927144663026507
CUST-01238,residential,NBN100,month-to-month,Brisbane,1,77.68,17.534777777777773,15.468888888888891,8.778888888888888,0.3303333333333333,0.35555555555555557,2.0,1.0,1.0,1.0,1.0,12.0,77.77083333333333,0.08333333333333333,0.08333333333333333,0,0.04202612159896645
CUST-01084,residential,NBN250,2-year,Melbourne,11,100.14,37.89533333333333,11.911111111111111,6.736666666666666,0.40777777777777774,0.35555555555555557,3.0,2.0,3.0,0.6666666666666666,4.5,12.0,99.62333333333333,0.08333333333333333,0.0,0,0.04345363469363999
CUST-01395,wholesale,Fibre,2-year,Brisbane,13,297.53,152.76900000000003,14.937777777777773,8.597777777777777,0.39566666666666667,0.37777777777777777,2.0,2.0,2.0,1.0,5.0,12.0,296.9883333333334,0.0,0.0,0,0.004180083354007641
CUST-01990,residential,NBN50,2-year,Brisbane,30,57.4,8.469777777777777,14.80111111111111,8.016666666666667,0.4485555555555556,0.4111111111111111,2.0,1.0,3.0,1.0,3.3333333333333335,12.0,57.845,0.0,0.0,0,0.009101191033806197
